In [ ]:
import threading
import cv2 
import time
import math
from time import sleep
import ipywidgets.widgets as widgets
from gesture_action import handDetector
from jetcobot_utils.grasp_controller import GraspController
import time

In [ ]:
graspController = GraspController()
graspController.init_pose()

In [ ]:
g_camera = cv2.VideoCapture(0, cv2.CAP_V4L)
g_camera.set(3, 640)
g_camera.set(4, 480)
g_camera.set(5, 30)  
g_camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M', 'J', 'P', 'G'))
g_camera.set(cv2.CAP_PROP_BRIGHTNESS, 40) 
g_camera.set(cv2.CAP_PROP_CONTRAST, 50) 
g_camera.set(cv2.CAP_PROP_EXPOSURE, 156) 

In [ ]:
hand_detector = handDetector(detectorCon=0.75)
image_original = widgets.Image(format='jpeg', width=640, height=480)
image_result = widgets.Image(format='jpeg', width=640, height=480)
image_widget = widgets.HBox([image_original, image_result])

In [ ]:
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

In [ ]:
display(image_widget)
m_fps = 0
t_start = time.time()
try:
    while True:
        ret, frame = g_camera.read()
        frame, img = hand_detector.findHands(frame, draw=False)
        if len(hand_detector.lmList) != 0:
            finger_number = hand_detector.get_gesture()
            cv2.rectangle(frame, (0, 430), (230, 480), (0, 255, 0), cv2.FILLED)
            cv2.putText(frame, str(finger_number), (10, 470), cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 0), 2)
            print("Number:", finger_number)
        
        m_fps = m_fps + 1
        fps = m_fps / (time.time() - t_start)
        if (time.time() - t_start) >= 2:
            m_fps = fps
            t_start = time.time() - 1
        text="FPS:" + str(int(fps))
        cv2.putText(frame, text, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        image_original.value = bgr8_to_jpeg(frame)
        image_result.value = bgr8_to_jpeg(img)
except:
    print(" Program closed! ")
    pass


In [ ]:
g_camera.release()             